# Inferencia con Qwen3-VL-8B-Instruct — Entrada: imagen + texto

## Importar librerías

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
import json
import os
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

import transformers
# Qwen3-VL usa su propia clase de modelo
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_curve,
    auc
)

from pyevall.evaluation import PyEvALLEvaluation
from pyevall.metrics.metricfactory import MetricFactory

# Mostrar versiones para debugging
print(f"Transformers version: {transformers.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Transformers version: 5.3.0
PyTorch version: 2.9.1+cu128
CUDA disponible: True
GPU: NVIDIA L40S
Memoria GPU: 47.7 GB


In [2]:
# Verificar dependencias para Qwen3-VL
import subprocess
import sys

packages_to_install = []

try:
    from packaging import version
    print("✓ packaging ya está instalado")
except ImportError:
    packages_to_install.append("packaging")

try:
    import qwen_vl_utils
    print("✓ qwen-vl-utils ya está instalado")
except ImportError:
    packages_to_install.append("qwen-vl-utils")

if packages_to_install:
    print(f"Instalando paquetes: {', '.join(packages_to_install)}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages_to_install + ["-q"])
    print(f"✓ Paquetes instalados correctamente: {', '.join(packages_to_install)}")
else:
    print("✓ Todas las dependencias están instaladas")

✓ packaging ya está instalado
✓ qwen-vl-utils ya está instalado
✓ Todas las dependencias están instaladas


## Configuración y parámetros

In [3]:
# ====================
# CONFIGURACIÓN GLOBAL
# ====================

MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
MAIN_PATH  = ".."
GROUP_ID   = "BeingChillingWeWillWin"
MODEL_ID   = "qwen3_vl_8b"

TEXT_COLUMN  = "combined_text"
LABEL_COLUMN = "label"

DATA_TRAIN_PATH = os.path.join(MAIN_PATH, "preprocessed_data", "train_split.json")
DATA_VAL_PATH   = os.path.join(MAIN_PATH, "preprocessed_data", "dev_split.json")
DATA_TEST_PATH  = os.path.join(MAIN_PATH, "preprocessed_data", "test_split.json")

DATA_BASE_DIR   = os.path.join(MAIN_PATH, "materials", "dataset_task2_exist2026")
PREDICTIONS_DIR = os.path.join(MAIN_PATH, "predictions")
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

MAX_NEW_TOKENS = 128
TEMPERATURE    = 0.3
TOP_P          = 0.9

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

label_map         = {"NO": 0, "YES": 1}
label_map_inverse = {0: "NO", 1: "YES"}

## Carga y preprocesamiento de datos

In [4]:
def load_json_dataset(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return pd.DataFrame(data.values())

def build_combined_text(row):
    img_desc = str(row.get('image_description', '') or '')
    txt      = str(row.get('text', '') or '')
    return f"descripcion imagen: {img_desc}. Texto: {txt}"

train_df = load_json_dataset(DATA_TRAIN_PATH)
val_df   = load_json_dataset(DATA_VAL_PATH)
test_df  = load_json_dataset(DATA_TEST_PATH)

for df in [train_df, val_df, test_df]:
    df[TEXT_COLUMN] = df.apply(build_combined_text, axis=1)

train_df["label_int"] = train_df[LABEL_COLUMN].map(label_map)
val_df["label_int"]   = val_df[LABEL_COLUMN].map(label_map)
test_df["label_int"]  = -1

print(f"Train size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")
print(f"\nEjemplo de entrada:\n  {train_df[TEXT_COLUMN].iloc[0][:200]}")
print(f"\nDistribución de etiquetas en TRAIN:")
print(train_df[LABEL_COLUMN].value_counts())
print(f"\nDistribución de etiquetas en VAL:")
print(val_df[LABEL_COLUMN].value_counts())

Train size: 2146 | Val size: 537 | Test size: 687

Ejemplo de entrada:
  descripcion imagen: a close up of a snake with its mouth open and its tongue out. Texto: Demostración de que las cosas mas peligrosas del mundo tienen el mismo aspecto. mémenoides 

Distribución de etiquetas en TRAIN:
label
YES    1282
NO      864
Name: count, dtype: int64

Distribución de etiquetas en VAL:
label
YES    321
NO     216
Name: count, dtype: int64


## Carga del modelo Qwen3-VL-8B-Instruct

In [5]:
import torch
import gc
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

# Limpiar memoria GPU antes de cargar
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

print(f"Cargando modelo {MODEL_NAME}...")

# Cargar modelo Qwen3-VL-8B-Instruct
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
)

# Cargar processor
processor = AutoProcessor.from_pretrained(MODEL_NAME)

# Verificar carga
print(f"✓ Modelo cargado correctamente")
print(f"✓ Processor cargado")
print(f"✓ Dispositivo: {next(model.parameters()).device}")
print(f"✓ Dtype: {model.dtype}")

Cargando modelo Qwen/Qwen3-VL-8B-Instruct...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

✓ Modelo cargado correctamente
✓ Processor cargado
✓ Dispositivo: cuda:0
✓ Dtype: torch.bfloat16


## Funciones de inferencia

In [6]:
import gc
from qwen_vl_utils import process_vision_info


def build_messages(image_path, combined_text):
    """Construye los mensajes en formato Qwen3-VL."""
    system_message = {
        "role": "system",
        "content": (
            "You are an expert content moderator specialized in identifying sexist content in memes. "
            "Your task is to analyze both the visual content and the text to determine if the meme "
            "contains sexist elements (stereotypes, discrimination, objectification, or derogatory content "
            "towards any gender)."
        )
    }
    
    user_message = {
        "role": "user",
        "content": [
            {"type": "image", "image": str(image_path)},
            {"type": "text", "text": (
                f"Analyze this meme carefully. {combined_text}\n\n"
                "Does this meme contain sexist content?\n\n"
                "Answer ONLY with 'YES' if it contains sexist content, or 'NO' if it doesn't.\n"
                "Format: CLASSIFICATION: [YES/NO]"
            )}
        ]
    }
    
    return [system_message, user_message]


def parse_model_response(response_text):
    """Parsea la respuesta del modelo para extraer la clasificación."""
    response_upper = response_text.upper()
    
    if "CLASSIFICATION: YES" in response_upper or "CLASSIFICATION:YES" in response_upper:
        classification = "YES"
    elif "CLASSIFICATION: NO" in response_upper or "CLASSIFICATION:NO" in response_upper:
        classification = "NO"
    elif response_upper.strip().startswith("YES"):
        classification = "YES"
    elif response_upper.strip().startswith("NO"):
        classification = "NO"
    else:
        classification = "NO"
        print(f"[WARN] No se pudo parsear: {response_text[:100]}")
    
    confidence = 0.9 if ("CLASSIFICATION:" in response_text) else 0.6
    return classification, confidence


@torch.no_grad()
def classify_image(image_path, combined_text, model, processor, max_retries=2):
    """Clasifica una imagen usando Qwen3-VL."""
    for attempt in range(max_retries):
        try:
            messages = build_messages(image_path, combined_text)
            
            # Preparar texto con template de chat
            text = processor.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=True
            )
            
            # Procesar visión para obtener imágenes y videos
            image_inputs, video_inputs = process_vision_info(messages)
            
            # Preparar inputs
            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt"
            ).to(model.device)
            
            # Generar respuesta
            generated_ids = model.generate(
                **inputs, 
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE if TEMPERATURE > 0 else None,
                top_p=TOP_P if TEMPERATURE > 0 else None,
                do_sample=TEMPERATURE > 0,
            )
            
            # Decodificar solo los nuevos tokens
            generated_ids_trimmed = [
                out_ids[len(in_ids):] 
                for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]
            response = processor.batch_decode(
                generated_ids_trimmed, 
                skip_special_tokens=True, 
                clean_up_tokenization_spaces=False
            )[0]
            
            # Limpiar memoria
            del inputs, generated_ids, generated_ids_trimmed
            torch.cuda.empty_cache()
            
            classification, confidence = parse_model_response(response)
            return {'classification': classification, 'confidence': confidence, 'raw_response': response}
            
        except torch.cuda.OutOfMemoryError:
            print(f"[OOM] {image_path.name if hasattr(image_path, 'name') else image_path}: Reintentando...")
            torch.cuda.empty_cache()
            gc.collect()
            continue
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"[WARN] {image_path}: {str(e)[:50]}... Reintentando...")
                torch.cuda.empty_cache()
                continue
            else:
                print(f"[ERROR] {image_path}: {str(e)[:100]}")
                return {'classification': 'NO', 'confidence': 0.0, 'raw_response': ''}
    
    return {'classification': 'NO', 'confidence': 0.0, 'raw_response': ''}


def process_dataset(df, base_dir, model, processor, split_name="dev"):
    """Procesa un dataset completo con limpieza periódica de memoria."""
    results = []
    base_path = Path(base_dir)
    
    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=f"Inferencia {split_name}")):
        img_path = base_path / row['path_memes']
        prediction = classify_image(img_path, row[TEXT_COLUMN], model, processor)
        
        result = {
            'id_EXIST': str(row['id_EXIST']),
            'classification': prediction['classification'],
            'confidence': prediction['confidence'],
        }
        
        if 'label_int' in row.index and row['label_int'] >= 0:
            result['true_label'] = label_map_inverse[row['label_int']]
        
        results.append(result)
        
        # Limpieza periódica de memoria
        if (idx + 1) % 50 == 0:
            torch.cuda.empty_cache()
            gc.collect()
    
    return results


def save_probs_json(ids, probs, split_name, labels=None):
    """Guarda probabilidades en formato JSON."""
    records = []
    for i, (id_exist, prob) in enumerate(zip(ids, probs)):
        rec = {'id': str(id_exist), 'prob_YES': round(float(prob), 6)}
        if labels is not None:
            rec['label'] = label_map_inverse[int(labels[i])]
        records.append(rec)
    path = os.path.join(PREDICTIONS_DIR, f'{GROUP_ID}_{MODEL_ID}_probs_{split_name}.json')
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

## Inferencia en DEV y evaluación

In [7]:
val_results = process_dataset(val_df, DATA_BASE_DIR, model, processor, split_name="dev")

y_pred_labels = [label_map[r['classification']] for r in val_results]
y_true_labels = [label_map[r['true_label']] for r in val_results]

y_probs_val = [r['confidence'] if r['classification'] == 'YES' else (1 - r['confidence']) for r in val_results]

# Métricas
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_score, recall_score

accuracy  = accuracy_score(y_true_labels, y_pred_labels)
f1        = f1_score(y_true_labels, y_pred_labels, pos_label=1)
precision = precision_score(y_true_labels, y_pred_labels, pos_label=1)
recall    = recall_score(y_true_labels, y_pred_labels, pos_label=1)

print(f"\n=== Resultados en DEV ({MODEL_ID}) ===")
print(f"  Accuracy : {accuracy:.4f}")
print(f"  F1-Score : {f1:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall   : {recall:.4f}")

print(f"\nClassification Report:")
print(classification_report(y_true_labels, y_pred_labels, target_names=['NO', 'YES']))

save_probs_json(val_df['id_EXIST'].values, y_probs_val, 'dev', y_true_labels)

Inferencia dev:   0%|          | 0/537 [00:00<?, ?it/s]


=== Resultados en DEV (qwen3_vl_8b) ===
  Accuracy : 0.7598
  F1-Score : 0.8254
  Precision: 0.7297
  Recall   : 0.9502

Classification Report:
              precision    recall  f1-score   support

          NO       0.87      0.48      0.61       216
         YES       0.73      0.95      0.83       321

    accuracy                           0.76       537
   macro avg       0.80      0.71      0.72       537
weighted avg       0.78      0.76      0.74       537



## Evaluación en DEV con PyEvALL

In [8]:
dev_preds_for_pyevall = [
    {'test_case': 'EXIST2025', 'id': str(r['id_EXIST']), 'value': r['classification']}
    for r in val_results
]
dev_preds_df   = pd.DataFrame(dev_preds_for_pyevall)
dev_preds_path = os.path.join(PREDICTIONS_DIR, 'dev_predictions_temp.json')
with open(dev_preds_path, 'w', encoding='utf-8') as f:
    f.write(dev_preds_df.to_json(orient='records'))

dev_gold = [
    {'test_case': 'EXIST2025', 'id': str(id_exist), 'value': label}
    for id_exist, label in zip(val_df['id_EXIST'].values, val_df[LABEL_COLUMN].values)
]
dev_gold_df   = pd.DataFrame(dev_gold)
dev_gold_path = os.path.join(PREDICTIONS_DIR, 'dev_gold_temp.json')
with open(dev_gold_path, 'w', encoding='utf-8') as f:
    f.write(dev_gold_df.to_json(orient='records'))

test_eval = PyEvALLEvaluation()
metrics   = [MetricFactory.Accuracy.value, MetricFactory.FMeasure.value]
report    = test_eval.evaluate(dev_preds_path, dev_gold_path, metrics)
print("\n=== Evaluación en DEV con PyEvALL ===")
report.print_report()

2026-03-06 01:29:35,082 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['Accuracy', 'FMeasure']


2026-03-06 01:29:35,114 - pyevall.metrics.metrics - INFO -             evaluate() - Executing accuracy evaluation method


2026-03-06 01:29:35,172 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method



=== Evaluación en DEV con PyEvALL ===
{
  "metrics": {
    "Accuracy": {
      "name": "Accuracy",
      "acronym": "Acc",
      "description": "Coming soon!",
      "status": "OK",
      "results": {
        "test_cases": [{
          "name": "EXIST2025",
          "average": 0.7597765363128491
        }],
        "average_per_test_case": 0.7597765363128491
      }
    },
    "FMeasure": {
      "name": "F-Measure",
      "acronym": "F1",
      "description": "Coming soon!",
      "status": "OK",
      "results": {
        "test_cases": [{
          "name": "EXIST2025",
          "classes": {
            "YES": 0.8254397834912043,
            "NO": 0.6149253731343284
          },
          "average": 0.7201825783127663
        }],
        "average_per_test_case": 0.7201825783127663
      }
    }
  },
  "files": {
    "dev_predictions_temp.json": {
      "name": "dev_predictions_temp.json",
      "status": "OK",
      "gold": false,
      "description": "Use parameter: report=\"embedd

## Inferencia en TEST y generación de predicciones finales

In [9]:
test_results = process_dataset(test_df, DATA_BASE_DIR, model, processor, split_name="test")

y_probs_test = [r['confidence'] if r['classification'] == 'YES' else (1 - r['confidence']) for r in test_results]
test_preds   = [r['classification'] for r in test_results]

save_probs_json(test_df['id_EXIST'].values, y_probs_test, 'test')

print(f"\nPredicciones en TEST:")
print(f"  Total: {len(test_preds)}")
print(f"  YES  : {sum(1 for p in test_preds if p == 'YES')} ({100*sum(1 for p in test_preds if p == 'YES')/len(test_preds):.2f}%)")
print(f"  NO   : {sum(1 for p in test_preds if p == 'NO')} ({100*sum(1 for p in test_preds if p == 'NO')/len(test_preds):.2f}%)")

Inferencia test:   0%|          | 0/687 [00:00<?, ?it/s]


Predicciones en TEST:
  Total: 687
  YES  : 522 (75.98%)
  NO   : 165 (24.02%)


## Guardar predicciones en formato PyEvALL para TEST

In [10]:
test_preds_for_submission = [
    {'test_case': 'EXIST2025', 'id': str(r['id_EXIST']), 'value': r['classification']}
    for r in test_results
]
test_preds_df = pd.DataFrame(test_preds_for_submission)

output_filename = f"{GROUP_ID}_{MODEL_ID}.json"
output_path     = os.path.join(PREDICTIONS_DIR, output_filename)

with open(output_path, 'w', encoding='utf-8') as f:
    f.write(test_preds_df.to_json(orient='records'))

print(f"\nPredicciones guardadas en: {output_path}")


Predicciones guardadas en: ../predictions/BeingChillingWeWillWin_qwen3_vl_8b.json
